# tensor-unbind composite — cx17: unbind (s,u,v) from a (NR, 3) solve result and classify intersections

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `tensor-unbind`, `triangle-barycentric`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "tensor-unbind"
DD_ATOM_IDS = ["tensor-unbind", "triangle-barycentric"]
DD_SUBTOPICS = ["Numpy: Indexing and selection", "Geometry: Barycentric coords"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA's ray-triangle intersection solves a 3x3 linear system per ray and gets back a `(NR, 3)` tensor where the three components are `(s, u, v)`: `s` is the parametric distance along the ray, and `(u, v)` are the **barycentric coordinates** locating the intersection inside the triangle.

The valid-intersection test combines barycentric facts:
  - `u >= 0`
  - `v >= 0`
  - `u + v <= 1`     (this is the triangle-barycentric atom — the simplex constraint)
  - `s >= 0`         (ray facing forward, not behind the camera)

To express this cleanly you `unbind` the `(NR, 3)` solve result along axis 1 to get three `(NR,)` tensors `s, u, v`, then combine the four boolean constraints. The named-vector unpack is what makes the test readable — without it you'd be writing `solve_result[:, 1] >= 0` four times.

### Composite Exercise — unbind (s,u,v) from a (NR, 3) solve result and classify intersections

**Atoms exercised together**: `tensor-unbind`, `triangle-barycentric`

Implement `cx17_classify_hits(solve_result)` where `solve_result: (NR, 3)` packs the per-ray `(s, u, v)` solution from `torch.linalg.solve`.

1. **Unbind** along axis 1 to get three `(NR,)` tensors: `s, u, v = t.unbind(solve_result, dim=1)`.
2. Apply the **barycentric** + ray-forward validity test elementwise:
   - `u >= 0`
   - `v >= 0`
   - `u + v <= 1`
   - `s >= 0`
3. Combine all four with `&` (boolean AND).

Return a boolean `(NR,)` tensor — `True` for valid intersections, `False` otherwise.

In [ ]:
def cx17_classify_hits(solve_result):
    # Atom A (tensor-unbind): extract the three named components from the (NR, 3) result.
    s, u, v = t.unbind(solve_result, dim=1)
    # Atom B (triangle-barycentric): the inside-triangle test is u>=0, v>=0, u+v<=1;
    # plus s>=0 ensures the hit is along the forward ray direction.
    return (u >= 0) & (v >= 0) & ((u + v) <= 1) & (s >= 0)


<details><summary>Show solution — cx17</summary>

```python
def cx17_classify_hits(solve_result):
    # Atom A (tensor-unbind): extract the three named components from the (NR, 3) result.
    s, u, v = t.unbind(solve_result, dim=1)
    # Atom B (triangle-barycentric): the inside-triangle test is u>=0, v>=0, u+v<=1;
    # plus s>=0 ensures the hit is along the forward ray direction.
    return (u >= 0) & (v >= 0) & ((u + v) <= 1) & (s >= 0)
```

The barycentric simplex constraint `u >= 0, v >= 0, u + v <= 1` describes the interior + boundary of the standard 2-simplex. Adding `s >= 0` filters out hits BEHIND the camera (the ray equation `O + s*D` with negative `s` points the wrong way). Unbind makes the four constraints readable as four short boolean ops; alternative `solve_result[:, 0] >= 0` style code obscures which component is which. Note the edge case `u + v == 1` (lying on edge BC) is INCLUDED — `<=` not `<`. Many bugs come from flipping that to strict inequality.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx17'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx17',
        'subtopics': ["Numpy: Indexing and selection", "Geometry: Barycentric coords"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()